# Hypertension Prediction - Training Notebook

This notebook documents the full ML workflow used in the project: data loading, cleaning, encoding, EDA, model training, evaluation, model selection, and serialization.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

sns.set_theme(style='whitegrid')

In [ ]:
DATA_PATH = 'patient_data.csv'
MODEL_PATH = '../logreg_model.pkl'

df = pd.read_csv(DATA_PATH)
df.head()

## 1. Data Cleaning and Normalization

In [ ]:
# Rename gender column from legacy source
if 'C' in df.columns:
    df = df.rename(columns={'C': 'Gender'})

# Trim spaces
for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].astype(str).str.strip()

# Fix known typos/variants
df['Severity'] = df['Severity'].replace({'Sever': 'Severe'})
df['Systolic'] = df['Systolic'].replace({'121- 130': '121-130', '111 - 120': '111-120'})
df['Diastolic'] = df['Diastolic'].replace({'70 - 80': '70-80', '81 - 90': '81-90', '91 - 100': '91-100'})
df['Stages'] = df['Stages'].replace({
    'HYPERTENSION (Stage-2).': 'HYPERTENSION (Stage-2)',
    'HYPERTENSIVE CRISI': 'HYPERTENSIVE CRISIS'
})

df = df[df['Stages'].isin(['NORMAL', 'HYPERTENSION (Stage-1)', 'HYPERTENSION (Stage-2)', 'HYPERTENSIVE CRISIS'])].copy()
df.shape

## 2. Label Encoding (Project Mapping)

In [ ]:
encoders = {
    'Gender': {'Male': 0, 'Female': 1},
    'Age': {'18-34': 1, '35-50': 2, '51-64': 3, '65+': 4},
    'History': {'No': 0, 'Yes': 1},
    'Patient': {'No': 0, 'Yes': 1},
    'TakeMedication': {'No': 0, 'Yes': 1},
    'Severity': {'Mild': 0, 'Moderate': 1, 'Severe': 2},
    'BreathShortness': {'No': 0, 'Yes': 1},
    'VisualChanges': {'No': 0, 'Yes': 1},
    'NoseBleeding': {'No': 0, 'Yes': 1},
    'Whendiagnoused': {'<1 Year': 1, '1 - 5 Years': 2, '>5 Years': 3},
    'Systolic': {'100-110': 0, '111-120': 1, '121-130': 2, '130+': 3},
    'Diastolic': {'70-80': 0, '81-90': 1, '91-100': 2, '100+': 3},
    'ControlledDiet': {'No': 0, 'Yes': 1}
}

target_map = {
    'NORMAL': 0,
    'HYPERTENSION (Stage-1)': 1,
    'HYPERTENSION (Stage-2)': 2,
    'HYPERTENSIVE CRISIS': 3
}

feature_cols = [
    'Gender', 'Age', 'History', 'Patient', 'TakeMedication', 'Severity',
    'BreathShortness', 'VisualChanges', 'NoseBleeding', 'Whendiagnoused',
    'Systolic', 'Diastolic', 'ControlledDiet'
]

encoded_df = df.copy()
for col, mp in encoders.items():
    encoded_df[col] = encoded_df[col].map(mp)

encoded_df['Target'] = encoded_df['Stages'].map(target_map)
encoded_df = encoded_df.dropna(subset=feature_cols + ['Target']).copy()
encoded_df['Target'] = encoded_df['Target'].astype(int)
encoded_df[feature_cols + ['Target']].head()

## 3. Exploratory Data Analysis

In [ ]:
# 3.1 Gender distribution
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x='Gender', palette='Set2', ax=ax[0])
ax[0].set_title('Gender Count Distribution')

df['Gender'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=ax[1])
ax[1].set_ylabel('')
ax[1].set_title('Gender Percentage')
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Stage distribution
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='Stages', order=['NORMAL', 'HYPERTENSION (Stage-1)', 'HYPERTENSION (Stage-2)', 'HYPERTENSIVE CRISIS'])
plt.xticks(rotation=20)
plt.title('Hypertension Stage Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Systolic vs Diastolic correlation (midpoint conversion)
sys_mid = {'100-110': 105, '111-120': 115.5, '121-130': 125.5, '130+': 135}
dia_mid = {'70-80': 75, '81-90': 85.5, '91-100': 95.5, '100+': 105}
corr_df = pd.DataFrame({
    'Systolic_mid': df['Systolic'].map(sys_mid),
    'Diastolic_mid': df['Diastolic'].map(dia_mid)
}).dropna()

plt.figure(figsize=(5, 4))
sns.heatmap(corr_df.corr(), annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation: Systolic vs Diastolic')
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 TakeMedication vs Severity
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='TakeMedication', hue='Severity', order=['No', 'Yes'])
plt.title('TakeMedication vs Severity')
plt.tight_layout()
plt.show()

In [ ]:
# 3.5 Age group vs stage
plt.figure(figsize=(10, 4))
sns.countplot(data=df, x='Age', hue='Stages', order=['18-34', '35-50', '51-64', '65+'])
plt.title('Age Group vs Hypertension Stage')
plt.tight_layout()
plt.show()

In [ ]:
# 3.6 Pairplot (midpoint numeric views)
pair_df = pd.DataFrame({
    'Systolic_mid': df['Systolic'].map(sys_mid),
    'Diastolic_mid': df['Diastolic'].map(dia_mid),
    'Stages': df['Stages']
}).dropna()
sns.pairplot(pair_df, hue='Stages', corner=True, diag_kind='kde')
plt.show()

## 4. Train/Test Split and MinMax Scaling

In [ ]:
X = encoded_df[feature_cols].copy()
y = encoded_df['Target'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set: {len(X_train)} samples ({len(X_train)/len(X):.0%})')
print(f'Testing set: {len(X_test)} samples ({len(X_test)/len(X):.0%})')

In [ ]:
# Apply MinMaxScaler to ordinal features
ordinal_cols = ['Age', 'Severity', 'Whendiagnoused', 'Systolic', 'Diastolic']

scaler = MinMaxScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[ordinal_cols] = scaler.fit_transform(X_train[ordinal_cols])
X_test_scaled[ordinal_cols] = scaler.transform(X_test[ordinal_cols])

X_train_scaled.head()

## 5. Model Training - Seven Algorithms

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1500),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'Ridge Classifier': RidgeClassifier(random_state=42),
    'Gaussian Naive Bayes': GaussianNB()
}

results = []
trained = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)

    row = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, pred, average='weighted', zero_division=0),
        'F1': f1_score(y_test, pred, average='weighted', zero_division=0)
    }
    results.append(row)
    trained[name] = model

results_df = pd.DataFrame(results).sort_values(by='F1', ascending=False).reset_index(drop=True)
results_df

In [ ]:
# Overfitting signal flag for perfect-accuracy models
results_df['Possible_Overfitting'] = np.where(results_df['Accuracy'] >= 1.0, 'High', 'Lower')
results_df

## 6. Model Selection (Logistic Regression for Deployment)

Even if some models produce perfect scores on this split, Logistic Regression is selected for deployment for better interpretability and reduced overfitting risk in real-world data.

In [ ]:
selected_name = 'Logistic Regression'
selected_model = trained[selected_name]

y_pred = selected_model.predict(X_test_scaled)
print('Selected Model:', selected_name)
print(classification_report(y_test, y_pred, digits=3))

## 7. Serialization for Deployment

In [ ]:
artifact = {
    'model': selected_model,
    'scaler': scaler,
    'ordinal_cols': ordinal_cols,
    'feature_cols': feature_cols,
    'encoders': encoders,
    'target_map': target_map,
    'inverse_target_map': {v: k for k, v in target_map.items()},
    'results_table': results_df
}

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(artifact, f)

print('Saved model artifact to:', os.path.abspath(MODEL_PATH))

In [ ]:
# Quick inference example
sample = {
    'Gender': 'Male', 'Age': '51-64', 'History': 'Yes', 'Patient': 'Yes',
    'TakeMedication': 'Yes', 'Severity': 'Moderate', 'BreathShortness': 'Yes',
    'VisualChanges': 'No', 'NoseBleeding': 'Yes', 'Whendiagnoused': '1 - 5 Years',
    'Systolic': '121-130', 'Diastolic': '91-100', 'ControlledDiet': 'No'
}

x = pd.DataFrame([sample])
for c, mp in encoders.items():
    x[c] = x[c].map(mp)
x[ordinal_cols] = scaler.transform(x[ordinal_cols])

pred = selected_model.predict(x[feature_cols])[0]
proba = selected_model.predict_proba(x[feature_cols])[0]
inv = {v: k for k, v in target_map.items()}

print('Predicted Stage:', inv[int(pred)])
print('Confidence (%):', round(float(np.max(proba) * 100), 2))